# Case study: differentiating a physics simulator (projectile with drag)

A projectile under gravity **and quadratic air drag** has no closed-form
trajectory — you integrate the equations of motion step by step. Once you are
simulating, natural questions are *sensitivities* (how does the range depend on
the launch angle, the speed, the drag coefficient?) and *design* (what launch
angle hits a target?). Both are gradients of the simulator's output with
respect to its inputs.

The usual answer is finite differences: nudge an input, re-simulate, subtract.
**Tangent** differentiates the time-stepping loop directly, giving exact
sensitivities and a readable gradient — and lets you optimize the launch by
gradient descent *through the simulator*.


In [ ]:
# Colab setup: install Tangent from PyPI (distributed as `tangent-ad`,
# imported as `tangent`). A no-op if it is already installed.
!pip install -q tangent-ad

In [1]:
import numpy as np
import tangent


## The simulator (plain NumPy)

Semi-implicit Euler integration of `v' = -g e_y - b |v| v`, `x' = v`. The drag
term `b |v| v` is nonlinear (it couples the velocity components through the
speed `|v|`), so there is nothing to differentiate by hand — but it is ordinary
NumPy that Tangent transforms as written. Shapes are scalars evolving in the
loop; `np.sqrt`, `np.sin`, `np.cos` and arithmetic are all Tangent handles.

In [2]:
def x_at_T(angle, v0, b, g, dt, n_steps):
    """Horizontal distance travelled after time T = n_steps * dt."""
    vx = v0 * np.cos(angle)
    vy = v0 * np.sin(angle)
    x = 0.0
    y = 0.0
    for step in range(n_steps):
        speed = np.sqrt(vx * vx + vy * vy)   # nonlinear drag couples vx, vy
        vx = vx + (-b * speed * vx) * dt
        vy = vy + (-g - b * speed * vy) * dt
        x = x + vx * dt
        y = y + vy * dt
    return x


## Simulate

In [3]:
g, dt, n_steps = 9.81, 0.01, 300      # T = 3.0 s
angle, v0, b = 0.70, 30.0, 0.02        # radians, m/s, drag coefficient
print("x(T) = %.4f m" % x_at_T(angle, v0, b, g, dt, n_steps))


x(T) = 41.1710 m


## Exact sensitivities in one pass

`tangent.grad(..., wrt=(0, 1, 2))` returns the derivative of the range with
respect to the launch angle, the launch speed, and the drag coefficient — all
at once.

In [4]:
dfun = tangent.grad(x_at_T, wrt=(0, 1, 2))
d_angle, d_v0, d_b = dfun(angle, v0, b, g, dt, n_steps)
print("dx/dangle = %+.5f  m/rad" % d_angle)
print("dx/dv0    = %+.5f  m/(m/s)" % d_v0)
print("dx/db     = %+.5f  m/(drag unit)" % d_b)


dx/dangle = -28.76882  m/rad
dx/dv0    = +0.87615  m/(m/s)
dx/db     = -755.70485  m/(drag unit)


## Validate against finite differences

In [5]:
h = 1e-6
def fd(i, *vals):
    up = list(vals); dn = list(vals)
    up[i] += h; dn[i] -= h
    return (x_at_T(*up, g, dt, n_steps) - x_at_T(*dn, g, dt, n_steps)) / (2 * h)

print("AD:", np.round([d_angle, d_v0, d_b], 6))
print("FD:", np.round([fd(0, angle, v0, b), fd(1, angle, v0, b), fd(2, angle, v0, b)], 6))


AD: [ -28.768818    0.876154 -755.704849]
FD: [ -28.768818    0.876154 -755.704849]


## Optimize the launch: gradient descent *through* the simulator

To land at a chosen distance, minimize `(x(T) - target)^2` over the launch
angle. The gradient of that loss w.r.t. the angle comes straight from Tangent,
so plain gradient descent steers the shot onto the target.

In [6]:
def miss(angle, v0, b, g, dt, n_steps, target):
    d = x_at_T(angle, v0, b, g, dt, n_steps) - target
    return d * d

target = 45.0
d_miss = tangent.grad(miss, wrt=(0,))
a, lr = 0.50, 1e-4
for it in range(61):
    a = a - lr * d_miss(a, v0, b, g, dt, n_steps, target)
    if it % 20 == 0:
        print("it %2d  angle = %.4f rad (%.1f deg)  ->  x(T) = %.3f m"
              % (it, a, np.degrees(a), x_at_T(a, v0, b, g, dt, n_steps)))
print("landed at %.3f m (target %.1f m)" % (x_at_T(a, v0, b, g, dt, n_steps), target))


it  0  angle = 0.5035 rad (28.9 deg)  ->  x(T) = 45.870 m


it 20  angle = 0.5391 rad (30.9 deg)  ->  x(T) = 45.160 m
it 40  angle = 0.5456 rad (31.3 deg)  ->  x(T) = 45.025 m


it 60  angle = 0.5466 rad (31.3 deg)  ->  x(T) = 45.004 m
landed at 45.004 m (target 45.0 m)


## The gradient is readable Python

The generated backward pass is an explicit reverse loop over the time steps —
ordinary NumPy you can read and step through.

In [7]:
src = dfun.__tangent_source__
print("\n".join(src.splitlines()[:22]))
print("...  (%d lines total)" % len(src.splitlines()))


def dx_at_Tdanglev0b(angle, v0, b, g, dt, n_steps, bx=1.0):
    # Initialize the tape
    _stack = tangent.Stack()
    vy_times_dt = None
    vx_times_dt = None
    _vy = None
    _vy2 = None
    _vy3 = None
    b_times_speed = None
    _vx = None
    _vx2 = None
    _vx3 = None
    minus_b = None
    speed = None
    _speed = None
    vx_times_vx = None
    vy_times_vy = None
    # Beginning of forward pass
    'Horizontal distance travelled after time T = n_steps * dt.'
    np_cos_angle = np.cos(angle)
    vx = v0 * np_cos_angle
    np_sin_angle = np.sin(angle)
...  (231 lines total)


## Takeaways

- A nonlinear ODE integrator (quadratic drag, no closed form) was
  differentiated **as written** — no rewrite into a framework, no hand-derived
  adjoint.
- Exact sensitivities to every launch parameter in **one reverse pass**,
  matching finite differences.
- The same gradient drives **optimization through the simulator** (steer the
  shot onto a target), and the adjoint is legible source you can inspect with
  `tangent.explain` / `tangent.source_map`.
